# Analysis of Process 31 Correlation Distribution across Major Bays

This notebook extracts the aggregated correlation data (median Pearson $r$ and significant station counts) for **Seto Inland Sea**, **Tokyo Bay**, and **Ise Bay**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Configure Pandas to show ALL rows without truncating
pd.set_option('display.max_rows', None)

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from step5_environmental_factor_correlation_analysis.process31_correlation_distribution import (
    load_correlation_seasons, build_long_table, count_significant
)
from utils.paths import ProjectPaths

def get_summary_table(target_region, season, layer, data_type):
    subset_dir = ProjectPaths.get_subset_dir(season, layer)
    pc1_path = subset_dir / f'pca_{data_type}_corr_pc1.csv'
    pc2_path = subset_dir / f'pca_{data_type}_corr_pc2.csv'
    
    corr_df_pc1 = pd.read_csv(pc1_path, dtype={'zettaicode': str}).set_index('zettaicode')
    corr_df_pc2 = pd.read_csv(pc2_path, dtype={'zettaicode': str}).set_index('zettaicode')
    
    merged_df = pd.read_feather(ProjectPaths.MERGED_DATA_FILE, columns=['zettaicode', 'watermass_name'])
    merged_df['zettaicode'] = merged_df['zettaicode'].astype(str).str.zfill(8)
    
    if target_region:
        valid_stations = merged_df[merged_df['watermass_name'] == target_region]['zettaicode'].unique()
        corr_df_pc1 = corr_df_pc1[corr_df_pc1.index.isin(valid_stations)]
        corr_df_pc2 = corr_df_pc2[corr_df_pc2.index.isin(valid_stations)]

    seasons = load_correlation_seasons()
    long_tables = {
        'PC1': build_long_table(corr_df_pc1, seasons),
        'PC2': build_long_table(corr_df_pc2, seasons),
    }
    
    results = []
    for pc in ['PC1', 'PC2']:
        df = long_tables[pc]
        for p_season in df['Param_Season'].unique():
            df_s = df[df['Param_Season'] == p_season].copy()
            if df_s.empty: continue
            medians = df_s.groupby('Parameter')['Pearson_r'].median()
            counts = count_significant(df_s)
            for param in medians.index:
                pos = int(counts.loc[(param, p_season), 'Positive']) if not counts.empty and (param, p_season) in counts.index else 0
                neg = int(counts.loc[(param, p_season), 'Negative']) if not counts.empty and (param, p_season) in counts.index else 0
                
                if pos > 0 or neg > 0:  # Filter out parameters with NO significant correlations
                    results.append({
                        'PC': pc,
                        'Param Season': p_season,
                        'Parameter': param,
                        'Median r': round(medians[param], 3),
                        'Sig Pos (>0)': pos,
                        'Sig Neg (<0)': neg,
                        'Total Sig': pos + neg
                    })
    
    if not results:
        return pd.DataFrame()
    
    res_df = pd.DataFrame(results).sort_values(['PC', 'Param Season', 'Total Sig'], ascending=[True, True, False]).drop(columns=['Total Sig'])
    return res_df.set_index(['PC', 'Param Season', 'Parameter'])

def analyze_region(region):
    display(HTML(f"<h2 style='color: #2e6c80; border-bottom: 2px solid #2e6c80; padding-bottom: 5px;'>{region}</h2>"))
    display(HTML("<h3>Environmental Factors</h3>"))
    display(get_summary_table(region, 'Summer', 'bottom', 'env'))
    display(HTML("<h3>Nutrient Factors</h3>"))
    display(get_summary_table(region, 'Summer', 'bottom', 'nutr'))


In [9]:
for region in ["Seto Inland Sea", "Tokyo Bay", "Ise Bay"]:
    analyze_region(region)

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                            
PC1 Autumn       Surface WTEMP       0.288            58             0
                 Bottom WTEMP        0.282            54             1
                 Bottom DENSITY     -0.256             0            52
                 Surface DENSITY    -0.254             0            50
                 Air Temp            0.177            37             1
                 Surface CHLA       -0.128             1            31
                 Wind Speed         -0.045             9            22
                 Bottom SALT        -0.211             0            26
                 Surface SALT       -0.184             0            19
                 Bottom CHLA        -0.132             1            16
                 Stratification      0.128            16             0
    Spring       Surface CHLA       -0.300             0            50
                 Bottom SALT        -0.202             6            33
                 Bottom DENSITY     -0.157             3            33
                 Air Temp            0.074            24             5
                 Bottom CHLA        -0.192             0            25
                 Surface WTEMP       0.178            20             1
                 Surface DENSITY    -0.101             2            18
                 Wind Speed         -0.079             6            12
                 Bottom WTEMP        0.126            16             0
                 Surface SALT       -0.083             5            11
                 Stratification     -0.014             8             2
    Summer       Surface CHLA       -0.204             2            30
                 Bottom DENSITY     -0.077             2            19
                 Stratification      0.129            15             6
                 Bottom CHLA        -0.177             0            20
                 Bottom SALT        -0.069             6            14
                 Bottom WTEMP        0.087             8             4
                 Surface DENSITY    -0.055             1            11
                 Wind Speed         -0.102             1             9
                 Air Temp           -0.039             5             4
                 Surface SALT       -0.040             1             8
                 Surface WTEMP       0.080             6             1
    Winter       Bottom DENSITY     -0.360             0            75
                 Surface WTEMP       0.338            74             0
                 Bottom WTEMP        0.326            66             0
                 Surface DENSITY    -0.331             0            65
                 Bottom SALT        -0.279             0            53
                 Surface SALT       -0.244             0            50
                 Wind Speed         -0.217             7            29
                 Bottom CHLA         0.003             9            14
                 Air Temp            0.066            15             5
                 Surface CHLA        0.015             7             9
                 Stratification      0.043             4             2
PC2 Autumn       Bottom WTEMP       -0.196             0            22
                 Surface WTEMP      -0.180             0            19
                 Air Temp           -0.176             0            17
                 Stratification      0.076             8             0
                 Bottom SALT        -0.127             0             7
                 Wind Speed         -0.052             3             3
                 Bottom CHLA         0.055             3             1
                 Surface SALT       -0.103             0             4
                 Bottom DENSITY      0.027             2             1
                 Surface CHLA       -0.011             0             2
                 Surface DENSITY     0.006             2  

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                          
PC1 Autumn       Bottom TN        -0.374             0            82
                 Surface TN       -0.361             0            80
                 Bottom DIN       -0.302             0            60
                 Surface DIN      -0.287             0            58
                 Surface COD       0.187            34             2
                 Bottom NO3-N     -0.217             1            30
                 Bottom COD        0.152            29             0
                 Bottom DIP        0.072            24             5
                 Bottom PO4-P      0.072            24             5
                 Surface NO3-N    -0.220             1            26
                 Surface TP       -0.091             4            22
                 Bottom TP        -0.045             6            18
                 Surface DIP      -0.049            14             9
                 Surface PO4-P    -0.049            14             9
                 Surface NH4-N    -0.008             4            12
                 Bottom NH4-N     -0.077             4             9
                 Surface DOC      -0.212             0            12
                 Surface NO2-N    -0.135             0            12
                 Bottom NO2-N     -0.089             0             7
    Spring       Surface TN       -0.399             0            76
                 Bottom TN        -0.360             1            69
                 Bottom DIN       -0.213             0            40
                 Bottom DIP        0.024            20             4
                 Bottom PO4-P      0.024            20             4
                 Surface TP       -0.145             3            21
                 Surface COD       0.036            18             3
                 Bottom TP        -0.141             6            14
                 Surface DIP      -0.007            16             4
                 Surface PO4-P    -0.007            16             4
                 Bottom COD        0.066            18             1
                 Bottom NH4-N     -0.052             3            15
                 Surface DIN      -0.068             1            17
                 Surface NH4-N     0.020             6            11
                 Bottom NO3-N     -0.008             5            11
                 Surface DOC      -0.178             0            15
                 Surface NO3-N     0.050             6             4
                 Surface NO2-N    -0.115             2             7
                 Bottom NO2-N     -0.106             3             4
    Summer       Surface TN       -0.348             0            76
                 Bottom TN        -0.338             0            66
                 Bottom DIN       -0.311             1            61
                 Bottom TP        -0.058            17            19
                 Bottom DIP        0.079            22            10
                 Bottom PO4-P      0.079            22            10
                 Bottom NO3-N     -0.219             0            31
                 Surface COD       0.073            16            14
                 Surface DOC      -0.258             0            30
                 Surface TP       -0.134             6            17
                 Bottom NO2-N     -0.127             3            19
                 Bottom COD        0.049            15             5
                 Surface DIN      -0.122             0            18
                 Surface DIP       0.012             9             6
                 Surface NH4-N     0.032             6             9
                 Surface PO4-P     0.012             9             6
                 Surface NO2-N    -0.125             2            10
                 Bottom NH4-N      0.001             5             5
                 Surface NO3-N

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                            
PC1 Autumn       Air Temp           -0.130             0             7
                 Surface WTEMP       0.094             5             0
                 Bottom CHLA        -0.046             0             2
                 Bottom DENSITY      0.026             1             1
                 Stratification      0.063             2             0
                 Surface DENSITY    -0.096             0             2
                 Surface SALT       -0.111             0             2
                 Bottom SALT         0.053             0             1
                 Bottom WTEMP        0.060             0             1
                 Surface CHLA        0.005             1             0
                 Wind Speed         -0.155             0             1
    Spring       Surface WTEMP       0.445            21             0
                 Bottom WTEMP        0.426            20             0
                 Stratification      0.367            12             0
                 Surface DENSITY    -0.279             0            10
                 Air Temp            0.287             7             0
                 Surface SALT       -0.150             0             6
                 Bottom SALT         0.096             3             0
                 Bottom DENSITY     -0.101             0             1
                 Wind Speed         -0.192             0             1
    Summer       Surface WTEMP       0.306            12             0
                 Wind Speed         -0.198             0             6
                 Stratification      0.204             5             0
                 Air Temp            0.215             4             0
                 Bottom CHLA        -0.160             0             4
                 Bottom WTEMP        0.162             4             0
                 Surface DENSITY    -0.208             0             2
                 Bottom DENSITY     -0.024             0             1
                 Surface CHLA       -0.023             0             1
    Winter       Bottom CHLA        -0.372             0            19
                 Surface CHLA       -0.360             0            17
                 Surface WTEMP       0.310            13             0
                 Bottom WTEMP        0.262             9             0
                 Surface DENSITY    -0.238             0             8
                 Bottom DENSITY     -0.139             1             5
                 Wind Speed         -0.262             0             5
                 Stratification      0.177             4             0
                 Air Temp            0.065             2             1
                 Surface SALT       -0.109             1             2
                 Bottom SALT         0.057             2             0
PC2 Autumn       Air Temp            0.040             2             3
                 Bottom SALT         0.137             4             0
                 Wind Speed         -0.076             0             4
                 Surface CHLA        0.164             3             0
                 Surface DENSITY     0.153             3             0
                 Bottom CHLA         0.096             2             0
                 Stratification     -0.096             0             2
                 Surface SALT        0.170             2             0
                 Bottom DENSITY      0.100             1             0
                 Bottom WTEMP        0.099             0             1
    Spring       Bottom WTEMP       -0.123             0             5
                 Bottom DENSITY      0.127             1             0
                 Stratification      0.081             1             0
                 Wind Speed          0.083             0             1
    Summer       Bottom DENSITY      0.097             4  

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                          
PC1 Autumn       Bottom TN        -0.360             1            16
                 Bottom TP        -0.334             0            15
                 Surface TN       -0.359             0            15
                 Bottom DIN       -0.307             0            13
                 Bottom NO3-N     -0.306             1            12
                 Surface DIN      -0.322             0            13
                 Surface NO3-N    -0.217             2            10
                 Surface TP       -0.289             0            12
                 Surface NO2-N    -0.280             0            11
                 Bottom DIP       -0.248             0            10
                 Bottom PO4-P     -0.248             0            10
                 Surface NH4-N    -0.155             0             7
                 Surface DIP      -0.186             0             6
                 Surface DOC       0.105             0             6
                 Surface PO4-P    -0.186             0             6
                 Bottom COD       -0.046             3             1
                 Bottom DOC       -0.656             0             4
                 Bottom NO2-N     -0.099             0             3
                 Bottom NH4-N     -0.118             0             1
                 Surface COD      -0.041             0             1
    Spring       Bottom NO3-N     -0.581             0            23
                 Bottom TN        -0.574             0            22
                 Surface DIN      -0.502             0            22
                 Surface NO2-N    -0.581             0            22
                 Surface NO3-N    -0.533             1            21
                 Surface TN       -0.534             0            22
                 Bottom DIN       -0.555             0            21
                 Bottom NO2-N     -0.345             0            15
                 Bottom TP        -0.254             0            11
                 Bottom NH4-N     -0.272             0             9
                 Surface NH4-N    -0.230             0             9
                 Surface TP       -0.212             0             6
                 Surface DOC       0.010             0             5
                 Bottom COD       -0.098             3             0
                 Bottom DIP       -0.092             0             2
                 Bottom PO4-P     -0.092             0             2
                 Bottom DOC       -0.440             0             1
                 Surface COD      -0.019             0             1
    Summer       Bottom TN        -0.459             0            22
                 Bottom DIN       -0.364             0            19
                 Surface TN       -0.427             0            19
                 Surface TP       -0.314             0            14
                 Surface NO2-N    -0.341             0            11
                 Surface DIP      -0.230             0            10
                 Surface PO4-P    -0.230             0            10
                 Bottom NO3-N     -0.240             0             9
                 Bottom TP        -0.240             0             9
                 Surface DIN      -0.231             0             9
                 Bottom NH4-N     -0.093             0             7
                 Surface DOC      -0.146             0             7
                 Bottom COD        0.011             4             2
                 Bottom NO2-N     -0.068             0             6
                 Surface NH4-N    -0.128             0             4
                 Bottom DIP       -0.153             0             2
                 Bottom PO4-P     -0.153             0             2
                 Bottom DOC        0.080             0             1
                 Surface NO3-N

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                            
PC1 Autumn       Air Temp            0.299            15             0
                 Bottom WTEMP        0.286            12             0
                 Surface WTEMP       0.212             9             0
                 Wind Speed         -0.088             0             6
                 Bottom DENSITY     -0.159             0             3
                 Bottom CHLA        -0.122             0             1
                 Bottom SALT        -0.045             0             1
                 Surface DENSITY    -0.172             0             1
                 Surface SALT       -0.115             0             1
    Spring       Air Temp            0.456            21             0
                 Surface WTEMP       0.259            11             0
                 Bottom WTEMP        0.241             8             0
                 Surface CHLA       -0.243             0             6
                 Surface DENSITY    -0.132             0             4
                 Stratification     -0.227             0             3
                 Wind Speed         -0.012             0             3
                 Surface SALT       -0.060             0             2
                 Bottom DENSITY     -0.027             0             1
    Summer       Air Temp            0.452            22             0
                 Surface WTEMP       0.383            21             0
                 Bottom WTEMP        0.249            12             0
                 Bottom CHLA        -0.191             0             8
                 Surface CHLA       -0.242             0             8
                 Bottom DENSITY     -0.091             1             4
                 Bottom SALT         0.016             1             3
                 Surface DENSITY    -0.025             0             3
                 Stratification     -0.198             0             2
                 Wind Speed          0.038             2             0
                 Surface SALT        0.016             0             1
    Winter       Bottom WTEMP        0.336            18             0
                 Surface WTEMP       0.284            13             0
                 Wind Speed         -0.045             0            11
                 Bottom CHLA        -0.202             0             9
                 Air Temp            0.198             4             1
                 Surface CHLA       -0.188             0             5
                 Stratification     -0.170             0             3
                 Bottom DENSITY     -0.169             0             2
                 Surface DENSITY    -0.029             1             1
PC2 Autumn       Bottom SALT        -0.086             0             7
                 Surface SALT       -0.097             0             6
                 Bottom DENSITY     -0.057             0             3
                 Stratification     -0.058             1             1
                 Surface DENSITY    -0.116             0             2
                 Surface CHLA       -0.034             0             1
                 Surface WTEMP      -0.007             0             1
    Spring       Bottom CHLA        -0.008             2             2
                 Surface CHLA        0.126             2             1
                 Bottom DENSITY      0.079             0             1
    Summer       Surface DENSITY    -0.246             0            10
                 Stratification      0.298             8             0
                 Air Temp            0.278             7             0
                 Surface WTEMP       0.261             7             0
                 Surface SALT       -0.193             0             4
                 Bottom DENSITY     -0.052             0             2
                 Bottom CHLA        -0.083             1  

Median r  Sig Pos (>0)  Sig Neg (<0)
PC  Param Season Parameter                                          
PC1 Autumn       Surface COD       0.267            14             0
                 Bottom DIN       -0.292             0            13
                 Bottom TN        -0.297             0            12
                 Surface DOC      -0.196             0            10
                 Bottom NH4-N     -0.251             0             8
                 Bottom NO3-N     -0.256             0             8
                 Surface TN       -0.230             0             8
                 Bottom COD        0.156             4             0
                 Surface NH4-N    -0.238             0             4
                 Bottom TP        -0.151             0             3
                 Surface DIN      -0.135             0             2
                 Bottom DIP       -0.032             1             0
                 Bottom PO4-P     -0.032             1             0
                 Surface NO2-N    -0.134             0             1
                 Surface TP       -0.137             0             1
    Spring       Surface TP       -0.344             0            19
                 Bottom TN        -0.320             0            17
                 Surface TN       -0.320             0            17
                 Bottom TP        -0.145             1            13
                 Surface DIN      -0.331             0            14
                 Bottom DIN       -0.291             0            12
                 Bottom COD        0.218            10             0
                 Surface NH4-N    -0.272             0             7
                 Bottom NO3-N     -0.222             0             6
                 Bottom NH4-N     -0.253             0             5
                 Surface COD       0.127             5             0
                 Surface NO2-N    -0.209             0             4
                 Surface NO3-N    -0.230             0             4
                 Bottom NO2-N     -0.169             0             3
                 Surface DIP      -0.174             0             3
                 Surface PO4-P    -0.174             0             3
                 Bottom DIP       -0.017             0             1
                 Bottom PO4-P     -0.017             0             1
                 Surface DOC      -0.050             0             1
    Summer       Surface DIN      -0.390             0            20
                 Surface TN       -0.352             0            20
                 Surface TP       -0.316             0            18
                 Bottom COD        0.186            16             0
                 Bottom DIN       -0.292             0            15
                 Bottom TN        -0.246             1            13
                 Bottom TP        -0.220             3            11
                 Bottom NO3-N     -0.259             0            11
                 Surface COD       0.178            11             0
                 Surface NO3-N    -0.357             0            11
                 Bottom DIP       -0.093             1             5
                 Bottom PO4-P     -0.093             1             5
                 Bottom NH4-N     -0.211             0             5
                 Bottom NO2-N     -0.149             0             5
                 Surface NH4-N    -0.170             0             5
                 Surface NO2-N    -0.378             0             4
                 Bottom DOC       -0.393             0             3
                 Surface DIP      -0.234             0             3
                 Surface PO4-P    -0.234             0             3
                 Surface DOC      -0.037             0             2
    Winter       Bottom TN        -0.436             0            23
                 Surface TN       -0.423             0            22
                 Bottom NH4-N 

## Interpretation of Seto Inland Sea

Based on the numerical summaries, we can interpret the biological and physical phenomena driving summer bottom Dissolved Oxygen (DO) variability in the Seto Inland Sea.

#### 1. Eutrophication is the Primary Driver of Oxygen Depletion (PC1)
The strongest signals in the entire dataset belong to **PC1's negative correlation with Nitrogen and Chlorophyll-a**.
*   **The Data:** PC1 has a massive number of stations with significant *negative* correlations to Surface TN, Bottom TN, Bottom DIN, and Surface CHLA.
*   **The Phenomenon (Organic Loading):** Assuming PC1 represents the primary variability of Bottom DO (where higher PC1 = higher DO), this negative relationship describes eutrophication. High levels of nitrogen fuel surface phytoplankton blooms (reflected by high Chlorophyll-a). When this massive amount of organic matter dies and sinks to the bottom, microbial decomposition consumes vast amounts of dissolved oxygen (DO). Therefore, higher Nitrogen and higher Chl-a directly correspond to lower bottom DO.

#### 2. Nitrogen-Limited System (N vs. P Dynamics)
*   **The Data:** While Nitrogen (TN, DIN) shows overwhelmingly negative correlations with DO, Phosphorus parameters (Bottom TP, Bottom PO4-P, Bottom DIP) are much more mixed, even leaning slightly towards *positive* correlations.
*   **The Phenomenon:** This strongly implies that the summer algal blooms driving the hypoxia in the Seto Inland Sea are **Nitrogen-limited**. Nitrogen is the primary dial controlling the severity of the organic loading and subsequent oxygen depletion, whereas Phosphorus dynamics are either uncoupled from this primary DO depletion cycle or driven by secondary factors.

#### 3. Freshwater Discharge and Physical Stratification (PC2)
*   **The Data:** PC2 is strongly characterized by its *negative* correlation with Salinity (Surface and Bottom) and Surface Density, while showing a *positive* correlation with Stratification.
*   **The Phenomenon (Riverine Influence):** This secondary mode clearly captures the physical influence of freshwater (river) discharge. High freshwater input drops the salinity (negative correlation). Because freshwater is lighter than seawater, it sits on top, strongly increasing density stratification (positive correlation). This stratification acts as a physical lid, preventing oxygen-rich surface water from mixing down to the bottom, thereby setting up the physical conditions necessary for hypoxia to develop.

#### Summary for Seto Inland Sea
The numerical summary captures a dual-threat mechanism for summer hypoxia: **PC2** sets the physical stage (freshwater input causing stratification that prevents oxygen mixing), while **PC1** provides the biochemical fuel (nitrogen-driven phytoplankton blooms that sink and rot, actively consuming the remaining oxygen).

## Interpretation of Tokyo Bay & Ise Bay

The data for these two highly enclosed bays reveals a profoundly different mechanism compared to the vast Seto Inland Sea system.

### 1. Severe Co-Limitation (Nitrogen AND Phosphorus)
Unlike the Seto Inland Sea (which was overwhelmingly driven by Nitrogen), both Tokyo Bay and Ise Bay show that **PC1 is strongly negatively correlated with BOTH Nitrogen and Phosphorus parameters**.
*   **Tokyo Bay:** Negative correlations across Bottom TN, Surface TN, Bottom DIN, Surface TP, Surface DIP, etc.
*   **Ise Bay:** Negative correlations across Surface DIN, Surface TN, Surface TP, Bottom DIN, Bottom TP, etc.
*   **Interpretation:** Both of these bays are highly enclosed and heavily urbanized. The data indicates that their eutrophication is so severe that both N and P act as major drivers of organic loading and subsequent bottom DO depletion. High nutrients across the board = low DO.

### 2. The Temperature & Eutrophication Paradox (PC1)
The most fascinating difference is how physical factors map to the DO variability in these enclosed bays.
*   **The Data:** In both Tokyo and Ise Bays, PC1 (which is negatively correlated with nutrients, meaning low PC1 = low DO) has a **strong positive correlation with Temperature** (Air Temp, Surface WTEMP, and Bottom WTEMP).
*   **The Phenomenon:** Normally, higher temperatures decrease oxygen solubility and increase respiration, making hypoxia *worse* (which would be a negative correlation). However, a positive correlation implies that **hotter summers are actually associated with HIGHER bottom DO** (or conversely, cooler summers have worse hypoxia).

**Why would this happen?**
1.  **River Discharge / Precipitation:** Cooler summers in Japan are often wetter (e.g., prolonged rainy seasons or typhoons). Massive rainfall drastically increases river discharge into these small bays, bringing immense loads of urban/agricultural nutrients and creating extreme density stratification (freshwater sitting on top). This triggers massive blooms and severe hypoxia.
2.  **Hotter = Drier:** Conversely, scorching, dry summers (high temperatures) mean less river discharge, lower nutrient runoff, and weaker freshwater-induced stratification, leading to less severe hypoxia.

### Overall Comparison
*   **Seto Inland Sea:** Primarily controlled by **Nitrogen** and the physical **density stratification** caused by river discharge (separated neatly into PC1 and PC2).
*   **Tokyo & Ise Bays:** Heavily controlled by **both Nitrogen & Phosphorus**. Their summer hypoxia severity appears to be dictated by the overarching climate/precipitation: hotter, drier years limit nutrient runoff and freshwater stratification, easing hypoxia, while cooler, wetter years dump nutrients and freshwater into the bays, exacerbating the oxygen depletion.